# AMP Peptide Generation

Generate peptides from trained organism-conditioned latent diffusion models (sample latent → scale by 1/25 → VAE decode).

Two paths:
1. **Initial models**: Phase 2 checkpoints trained **without** potency leveling
2. **Potency-leveled models**: Phase 2 checkpoints trained **with** Weak/Moderate/Strong level embeddings


### Project paths

This notebook lives in `notebooks/`. Shared code is under `src/`, data under `data/`.
On Colab, set `ROOT` to your cloned repo (or Drive copy) instead of `Path('..')`.


In [ ]:
import sys
from pathlib import Path

# Repo root (parent of notebooks/). On Colab, replace with your clone path, e.g.:
# ROOT = Path('/content/drive/MyDrive/your-thesis-repo')
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils.paths import DATA_DIR, CHECKPOINTS_DIR, RESULTS_DIR, PROJECT_ROOT
print('PROJECT_ROOT :', PROJECT_ROOT)
print('DATA_DIR     :', DATA_DIR)
print('CHECKPOINTS  :', CHECKPOINTS_DIR)
print('RESULTS      :', RESULTS_DIR)


## Generation - no potency leveling

Uses Phase 2 checkpoints trained on organism conditioning only.


In [ ]:
import os
import sys
import math
import random
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers.models.bert.modeling_bert import BertConfig, BertEncoder


PROJECT_ROOT = "/content/drive/MyDrive/DBAASP_Bioinformatics"
if PROJECT_ROOT.startswith("/content/drive") and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)

os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.models.transvae import create_VAE, org_dict, params, subsequent_mask, w2i

VAE_CHECKPOINT = f"{PROJECT_ROOT}/vae_weights.pth"
DIFFUSION_CHECKPOINT = f"{PROJECT_ROOT}/checkpoints_diffusion/phase2_2org/best_model.pth"
ORG_MAP_PATH = f"{PROJECT_ROOT}/latent_two_organism/cond_latents_new/organism_map.npy"
OUTPUT_DIR = f"{PROJECT_ROOT}/2organisms/generated_for_2organisms"

# Generation settings
NUM_SEQUENCES = 15000
BATCH_SIZE = 64
NUM_STEPS = None  # inferred from checkpoint when present; otherwise 500
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# Set GENERATE_ALL_ORGANISMS = False and ORGANISM_ID to generate for only want one organism
GENERATE_ALL_ORGANISMS = True
ORGANISM_ID = [0, 1]


DECODE_STRATEGY = "greedy"
TEMPERATURE = 1.0
TOP_K = 0
TOP_P = 1.0
MAX_DECODE_LEN = params["tgt_len"]

# Classifier-free guidance. 1.0 means normal conditional generation.
CFG_SCALE = 1.0
CLIP_DENOISED = False

LATENT_SCALE = 25.0
START_ID = w2i["<start>"]
EOS_ID = w2i["<end>"]
PAD_ID = w2i["_"]

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Device: {DEVICE}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Vocab OK: START={START_ID}, EOS={EOS_ID}, PAD={PAD_ID}")

def timestep_embedding(timesteps: torch.Tensor, dim: int, max_period: int = 10000) -> torch.Tensor:
    half = dim // 2
    freqs = torch.exp(
        -math.log(max_period) * torch.arange(start=0, end=half, dtype=torch.float32, device=timesteps.device) / half
    )
    args = timesteps[:, None].float() * freqs[None]
    embedding = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
    if dim % 2:
        embedding = torch.cat([embedding, torch.zeros_like(embedding[:, :1])], dim=-1)
    return embedding


class DModel(nn.Module):

    def __init__(self, num_class: int = 2, max_length: int = 1):
        super().__init__()
        self.input_channels = 128
        self.model_channels = 127
        self.out_channels = 128
        self.num_class = num_class
        self.drop_out = 0.1
        self.max_length = max_length

        config = BertConfig()
        config.hidden_dropout_prob = self.drop_out
        config.max_position_embeddings = self.max_length
        config.num_attention_heads = 6
        config.num_hidden_layers = 3
        config._attn_implementation = "eager"

        self.control_embedding = nn.Embedding(self.num_class, config.hidden_size)
        self.time_embed = nn.Sequential(
            nn.Linear(self.model_channels, config.hidden_size),
            nn.SiLU(),
            nn.Linear(config.hidden_size, config.hidden_size),
        )
        self.input_up_proj = nn.Sequential(
            nn.Linear(self.input_channels, config.hidden_size),
            nn.Tanh(),
            nn.Linear(config.hidden_size, config.hidden_size),
        )
        self.input_transformers = BertEncoder(config)
        self.register_buffer("position_ids", torch.arange(config.max_position_embeddings).expand((1, -1)))
        self.position_embeddings = nn.Embedding(config.max_position_embeddings, config.hidden_size)
        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.output_down_proj = nn.Sequential(
            nn.Linear(config.hidden_size, config.hidden_size),
            nn.Tanh(),
            nn.Linear(config.hidden_size, self.out_channels),
        )

    def forward(self, x: torch.Tensor, timesteps: torch.Tensor, control: Optional[torch.Tensor] = None) -> torch.Tensor:
        emb = self.time_embed(timestep_embedding(timesteps, self.model_channels))
        if control is not None:
            emb = emb + self.control_embedding(control)

        emb_x = self.input_up_proj(x)
        seq_length = x.size(1)
        pos_ids = self.position_ids[:, :seq_length]
        emb_inputs = self.position_embeddings(pos_ids) + emb_x + emb.unsqueeze(1).expand(-1, seq_length, -1)
        emb_inputs = self.dropout(self.LayerNorm(emb_inputs))

        attn_mask = torch.ones(x.shape[0], seq_length, device=x.device)
        extended_mask = attn_mask[:, None, None, :]
        extended_mask = (1.0 - extended_mask) * torch.finfo(emb_inputs.dtype).min
        h = self.input_transformers(emb_inputs, attention_mask=extended_mask)[0]
        return self.output_down_proj(h).type(x.dtype)


class CFGDenoiser(nn.Module):
    def __init__(self, model: nn.Module, cfg_scale: float, null_label: Optional[int]):
        super().__init__()
        self.model = model
        self.cfg_scale = cfg_scale
        self.null_label = null_label

    def forward(self, x: torch.Tensor, timesteps: torch.Tensor, control: Optional[torch.Tensor] = None) -> torch.Tensor:
        if control is None or self.cfg_scale == 1.0 or self.null_label is None:
            return self.model(x, timesteps, control)
        null_control = torch.full_like(control, self.null_label)
        cond_out = self.model(x, timesteps, control)
        uncond_out = self.model(x, timesteps, null_control)
        return uncond_out + self.cfg_scale * (cond_out - uncond_out)


def _extract_into_tensor(arr: np.ndarray, timesteps: torch.Tensor, broadcast_shape: Tuple[int, ...]) -> torch.Tensor:
    res = torch.from_numpy(arr).to(device=timesteps.device)[timesteps].float()
    while len(res.shape) < len(broadcast_shape):
        res = res[..., None]
    return res.expand(broadcast_shape)


class MyDiffusion:
    def __init__(self, num_timesteps: int, betas: np.ndarray):
        self.betas = betas
        alphas = 1.0 - betas
        self.num_timesteps = num_timesteps
        self.alphas_cumprod = np.cumprod(alphas, axis=0)
        self.alphas_cumprod_prev = np.append(1.0, self.alphas_cumprod[:-1])
        self.posterior_variance = betas * (1.0 - self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)
        self.posterior_log_variance_clipped = np.log(np.append(self.posterior_variance[1], self.posterior_variance[1:]))
        self.posterior_mean_coef1 = betas * np.sqrt(self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)
        self.posterior_mean_coef2 = (
            (1.0 - self.alphas_cumprod_prev) * np.sqrt(alphas) / (1.0 - self.alphas_cumprod)
        )

    def _scale_timesteps(self, t: torch.Tensor) -> torch.Tensor:
        return t.float() * (1000.0 / self.num_timesteps)

    def q_posterior_mean_variance(self, x_start: torch.Tensor, x_t: torch.Tensor, t: torch.Tensor):
        mean = (
            _extract_into_tensor(self.posterior_mean_coef1, t, x_t.shape) * x_start
            + _extract_into_tensor(self.posterior_mean_coef2, t, x_t.shape) * x_t
        )
        log_var = _extract_into_tensor(self.posterior_log_variance_clipped, t, x_t.shape)
        return mean, log_var

    def p_sample(self, model: nn.Module, x: torch.Tensor, t: torch.Tensor, cond=None, clip_denoised: bool = False):
        model_output = model(x, self._scale_timesteps(t), cond)
        pred_xstart = model_output.clamp(-1, 1) if clip_denoised else model_output
        mean, log_var = self.q_posterior_mean_variance(pred_xstart, x, t)
        noise = torch.randn_like(x)
        nonzero_mask = (t != 0).float().view(-1, *([1] * (len(x.shape) - 1)))
        return mean + nonzero_mask * torch.exp(0.5 * log_var) * noise

    @torch.no_grad()
    def p_sample_loop(self, model: nn.Module, shape: Tuple[int, ...], cond, device: torch.device, clip_denoised: bool = False):
        img = torch.randn(*shape, device=device)
        for i in tqdm(range(self.num_timesteps - 1, -1, -1), desc="diffusion", leave=False):
            t = torch.full((shape[0],), i, dtype=torch.long, device=device)
            img = self.p_sample(model, img, t, cond=cond, clip_denoised=clip_denoised)
        return img


def betas_for_alpha_bar(num_diffusion_timesteps: int, alpha_bar, max_beta: float = 0.999) -> np.ndarray:
    betas = []
    for i in range(num_diffusion_timesteps):
        t1 = i / num_diffusion_timesteps
        t2 = (i + 1) / num_diffusion_timesteps
        betas.append(min(1 - alpha_bar(t2) / alpha_bar(t1), max_beta))
    return np.array(betas)


def create_gaussian_diffusion(steps: int = 500) -> MyDiffusion:
    betas = betas_for_alpha_bar(steps, lambda t: 1 - np.sqrt(t + 0.0001))
    return MyDiffusion(num_timesteps=steps, betas=betas)

def strip_module_prefix(state_dict: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    return {k.replace("module.", "", 1): v for k, v in state_dict.items()}


def load_checkpoint(path: str, device: torch.device):
    checkpoint = torch.load(path, map_location=device)
    metadata = {}
    if isinstance(checkpoint, dict) and "model_state" in checkpoint:
        for key in ("num_organisms", "null_label", "num_steps"):
            if key in checkpoint:
                metadata[key] = int(checkpoint[key])
        checkpoint = checkpoint["model_state"]
    if not isinstance(checkpoint, dict):
        raise TypeError(f"Unsupported checkpoint format in {path}")
    return strip_module_prefix(checkpoint), metadata


def load_vae(path: str, device: torch.device) -> nn.Module:
    model = create_VAE()
    state_dict, _ = load_checkpoint(path, device)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return model


def ids_to_peptides(token_ids: torch.Tensor) -> List[str]:
    peptides = []
    for row in token_ids.detach().cpu().long().tolist():
        chars = []
        for token_id in row:
            if token_id in (EOS_ID, PAD_ID):
                break
            token = org_dict.get(int(token_id) - 1)
            if token is None or token == "<start>":
                continue
            chars.append(token)
        peptides.append("".join(chars))
    return peptides


def choose_next_token(logits: torch.Tensor, strategy: str, temperature: float, top_k: int, top_p: float) -> torch.Tensor:
    if strategy == "greedy":
        return torch.argmax(logits, dim=-1) + 1

    logits = logits / max(temperature, 1e-8)
    if top_k > 0:
        values, _ = torch.topk(logits, min(top_k, logits.shape[-1]), dim=-1)
        logits = logits.masked_fill(logits < values[:, [-1]], -float("inf"))

    if top_p < 1.0:
        sorted_logits, sorted_idx = torch.sort(logits, descending=True, dim=-1)
        sorted_probs = F.softmax(sorted_logits, dim=-1)
        cumulative = torch.cumsum(sorted_probs, dim=-1)
        remove = cumulative > top_p
        remove[:, 1:] = remove[:, :-1].clone()
        remove[:, 0] = False
        sorted_logits = sorted_logits.masked_fill(remove, -float("inf"))
        logits = torch.full_like(logits, -float("inf")).scatter(-1, sorted_idx, sorted_logits)

    probs = F.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1).squeeze(1) + 1


@torch.no_grad()
def decode_latents(vae: nn.Module, latents: torch.Tensor, src_mask: torch.Tensor) -> torch.Tensor:
    batch_size = latents.shape[0]
    decoded = torch.full((batch_size, 1), START_ID, dtype=torch.long, device=latents.device)
    output = torch.full((batch_size, MAX_DECODE_LEN), PAD_ID, dtype=torch.long, device=latents.device)
    finished = torch.zeros(batch_size, dtype=torch.bool, device=latents.device)

    for i in range(MAX_DECODE_LEN):
        tgt_mask = subsequent_mask(decoded.size(1)).to(device=latents.device, dtype=torch.bool)
        logits = vae.generator(vae.decode(latents, src_mask, decoded, tgt_mask))[:, i, :]
        next_token = choose_next_token(logits, DECODE_STRATEGY, TEMPERATURE, TOP_K, TOP_P)
        next_token = torch.where(finished, torch.full_like(next_token, PAD_ID), next_token)
        output[:, i] = next_token

        finished = finished | (next_token == EOS_ID) | (next_token == PAD_ID)
        decoded = torch.cat([decoded, next_token.unsqueeze(1)], dim=1)
        if bool(finished.all()):
            break

    return output


def predict_src_mask_from_latents(vae: nn.Module, latents: torch.Tensor) -> torch.Tensor:
    pred_len = vae.encoder.predict_len2(vae.encoder.predict_len1(latents))
    lengths = F.softmax(pred_len, dim=-1).argmax(dim=-1)
    lengths = lengths.clamp(min=1, max=params["src_len"] + 1)
    positions = torch.arange(params["src_len"] + 1, device=latents.device).unsqueeze(0)
    return (positions < lengths.unsqueeze(1)).unsqueeze(-2)


def prepare_vae_latents(vae: nn.Module, diffusion_sample: torch.Tensor):
    sample = diffusion_sample / LATENT_SCALE
    latents = sample.squeeze(1)
    if latents.ndim != 2 or latents.shape[-1] != params["d_latent"]:
        raise ValueError(f"Expected diffusion_clean sample shape (B, 1, 128), got {tuple(diffusion_sample.shape)}")
    src_mask = predict_src_mask_from_latents(vae, latents)
    return latents, src_mask


def batched(total: int, batch_size: int):
    for start in range(0, total, batch_size):
        yield min(batch_size, total - start)


def strip_module_prefix(state_dict: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    return {k.replace("module.", "", 1): v for k, v in state_dict.items()}


def load_checkpoint(path: str, device: torch.device):
    checkpoint = torch.load(path, map_location=device)
    metadata = {}
    if isinstance(checkpoint, dict) and "model_state" in checkpoint:
        for key in ("num_organisms", "null_label", "num_steps"):
            if key in checkpoint:
                metadata[key] = int(checkpoint[key])
        checkpoint = checkpoint["model_state"]
    if not isinstance(checkpoint, dict):
        raise TypeError(f"Unsupported checkpoint format in {path}")
    return strip_module_prefix(checkpoint), metadata


def load_vae(path: str, device: torch.device) -> nn.Module:
    model = create_VAE()
    state_dict, _ = load_checkpoint(path, device)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return model


def ids_to_peptides(token_ids: torch.Tensor) -> List[str]:
    peptides = []
    for row in token_ids.detach().cpu().long().tolist():
        chars = []
        for token_id in row:
            if token_id in (EOS_ID, PAD_ID):
                break
            token = org_dict.get(int(token_id) - 1)
            if token is None or token == "<start>":
                continue
            chars.append(token)
        peptides.append("".join(chars))
    return peptides


def choose_next_token(logits: torch.Tensor, strategy: str, temperature: float, top_k: int, top_p: float) -> torch.Tensor:
    if strategy == "greedy":
        return torch.argmax(logits, dim=-1) + 1

    logits = logits / max(temperature, 1e-8)
    if top_k > 0:
        values, _ = torch.topk(logits, min(top_k, logits.shape[-1]), dim=-1)
        logits = logits.masked_fill(logits < values[:, [-1]], -float("inf"))

    if top_p < 1.0:
        sorted_logits, sorted_idx = torch.sort(logits, descending=True, dim=-1)
        sorted_probs = F.softmax(sorted_logits, dim=-1)
        cumulative = torch.cumsum(sorted_probs, dim=-1)
        remove = cumulative > top_p
        remove[:, 1:] = remove[:, :-1].clone()
        remove[:, 0] = False
        sorted_logits = sorted_logits.masked_fill(remove, -float("inf"))
        logits = torch.full_like(logits, -float("inf")).scatter(-1, sorted_idx, sorted_logits)

    probs = F.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1).squeeze(1) + 1


@torch.no_grad()
def decode_latents(vae: nn.Module, latents: torch.Tensor, src_mask: torch.Tensor) -> torch.Tensor:
    batch_size = latents.shape[0]
    decoded = torch.full((batch_size, 1), START_ID, dtype=torch.long, device=latents.device)
    output = torch.full((batch_size, MAX_DECODE_LEN), PAD_ID, dtype=torch.long, device=latents.device)
    finished = torch.zeros(batch_size, dtype=torch.bool, device=latents.device)

    for i in range(MAX_DECODE_LEN):
        tgt_mask = subsequent_mask(decoded.size(1)).to(device=latents.device, dtype=torch.bool)
        logits = vae.generator(vae.decode(latents, src_mask, decoded, tgt_mask))[:, i, :]
        next_token = choose_next_token(logits, DECODE_STRATEGY, TEMPERATURE, TOP_K, TOP_P)
        next_token = torch.where(finished, torch.full_like(next_token, PAD_ID), next_token)
        output[:, i] = next_token

        finished = finished | (next_token == EOS_ID) | (next_token == PAD_ID)
        decoded = torch.cat([decoded, next_token.unsqueeze(1)], dim=1)
        if bool(finished.all()):
            break

    return output


def predict_src_mask_from_latents(vae: nn.Module, latents: torch.Tensor) -> torch.Tensor:
    pred_len = vae.encoder.predict_len2(vae.encoder.predict_len1(latents))
    lengths = F.softmax(pred_len, dim=-1).argmax(dim=-1)
    lengths = lengths.clamp(min=1, max=params["src_len"] + 1)
    positions = torch.arange(params["src_len"] + 1, device=latents.device).unsqueeze(0)
    return (positions < lengths.unsqueeze(1)).unsqueeze(-2)


def prepare_vae_latents(vae: nn.Module, diffusion_sample: torch.Tensor):
    sample = diffusion_sample / LATENT_SCALE
    latents = sample.squeeze(1)
    if latents.ndim != 2 or latents.shape[-1] != params["d_latent"]:
        raise ValueError(f"Expected diffusion_clean sample shape (B, 1, 128), got {tuple(diffusion_sample.shape)}")
    src_mask = predict_src_mask_from_latents(vae, latents)
    return latents, src_mask


def batched(total: int, batch_size: int):
    for start in range(0, total, batch_size):
        yield min(batch_size, total - start)


def load_models():
    device = torch.device(DEVICE)

    if SEED is not None:
        random.seed(SEED)
        np.random.seed(SEED)
        torch.manual_seed(SEED)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(SEED)

    print("Loading VAE...")
    vae = load_vae(VAE_CHECKPOINT, device)

    print("Loading diffusion model...")
    diffusion_state, metadata = load_checkpoint(DIFFUSION_CHECKPOINT, device)
    num_steps = NUM_STEPS or metadata.get("num_steps", 500)
    num_class = metadata.get("num_organisms", 1) + 1
    null_label = metadata.get("null_label")

    if "control_embedding.weight" in diffusion_state:
        num_class = diffusion_state["control_embedding.weight"].shape[0]

    diffusion_model = DModel(num_class=num_class, max_length=1).to(device)
    diffusion_model.load_state_dict(diffusion_state)
    diffusion_model.eval()

    denoiser = CFGDenoiser(diffusion_model, CFG_SCALE, null_label).to(device).eval()
    diffusion = create_gaussian_diffusion(num_steps)

    print(f"Loaded diffusion: steps={num_steps}, num_class={num_class}, null_label={null_label}")
    return vae, denoiser, diffusion, device


@torch.no_grad()
def generate_for_condition(vae, denoiser, diffusion, device, organism_id: int, num_sequences: int = NUM_SEQUENCES):
    peptides = []
    for current_batch in tqdm(list(batched(num_sequences, BATCH_SIZE)), desc=f"organism {organism_id}"):
        cond = torch.full((current_batch,), organism_id, dtype=torch.long, device=device)
        sample = diffusion.p_sample_loop(
            denoiser,
            (current_batch, 1, params["d_latent"]),
            cond=cond,
            device=device,
            clip_denoised=CLIP_DENOISED,
        )
        latents, src_mask = prepare_vae_latents(vae, sample)
        token_ids = decode_latents(vae, latents, src_mask)
        peptides.extend(ids_to_peptides(token_ids))
    return peptides[:num_sequences]


def save_peptides(peptides: List[str], output_path: str):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as handle:
        for peptide in peptides:
            handle.write(f"{peptide}\n")


def print_summary(name: str, peptides: List[str]):
    nonempty = [p for p in peptides if p]
    lengths = [len(p) for p in nonempty]
    print(f"\n{name}")
    print(f"  Generated: {len(peptides)}")
    print(f"  Nonempty : {len(nonempty)}")
    print(f"  Unique   : {len(set(nonempty))}")
    if lengths:
        print(f"  Lengths  : min={min(lengths)} max={max(lengths)} mean={np.mean(lengths):.2f} std={np.std(lengths):.2f}")
        print("  Preview  :")
        for peptide in nonempty[:5]:
            print(f"    {peptide} (len={len(peptide)})")

vae, denoiser, diffusion, device = load_models()

org_map = np.load(ORG_MAP_PATH, allow_pickle=True)
print(f"Organisms ({len(org_map)}):")
for i, name in enumerate(org_map):
    print(f"  [{i}] {name}")

all_results = {}
if GENERATE_ALL_ORGANISMS:
    organism_ids = list(range(len(org_map)))
else:
    organism_ids = [ORGANISM_ID]

for organism_id in organism_ids:
    organism_name = str(org_map[organism_id])
    print(f"\nGenerating for [{organism_id}] {organism_name}")
    peptides = generate_for_condition(vae, denoiser, diffusion, device, organism_id)
    all_results[organism_name] = peptides

    safe_name = organism_name.replace(" ", "_").replace("/", "-")
    output_path = f"{OUTPUT_DIR}/{safe_name}.txt"
    save_peptides(peptides, output_path)
    print_summary(organism_name, peptides)
    print(f"  Saved: {output_path}")

print("\nDone.")

## Generation - Potency-leveled models

Uses Phase 2 checkpoints trained **with** potency leveling (Weak / Moderate / Strong level embeddings + `cond_levels.npy`)


### Setup and configuration



In [ ]:
import os
import sys
import math
import random
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers.models.bert.modeling_bert import BertConfig, BertEncoder


PROJECT_ROOT = "/content/drive/MyDrive/DBAASP_Bioinformatics"

if PROJECT_ROOT.startswith("/content/drive") and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)

os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.models.transvae import create_VAE, org_dict, params, subsequent_mask, w2i

VAE_CHECKPOINT = f"{PROJECT_ROOT}/vae_weights.pth"
DIFFUSION_CHECKPOINT = f"{PROJECT_ROOT}/checkpoints_diffusion/phase2_leveledg/best_model.pth"

ORGANISM_MAP_PATH = f"{PROJECT_ROOT}/latent_withLevelings/cond_latents_new/organism_map.npy"
OUTPUT_DIR = f"{PROJECT_ROOT}/generatedPeptides/Strong"

# Generation settings
NUM_SEQUENCES = 1500
BATCH_SIZE = 64
NUM_STEPS = None  # inferred from checkpoint when present; otherwise 500
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Organism selection: condition id sent to the diffusion model
# Training order (must match checkpoint):
#   0 A. baumannii   1 B. subtilis   2 E. coli        3 E. faecalis
#   4 K. pneumoniae  5 P. aeruginosa 6 S. aureus      7 S. enterica
#   8 S. epidermidis

GENERATE_ALL_ORGANISMS = False
ORGANISM_ID = 1
# OR set by name instead of id (used when GENERATE_ALL_ORGANISMS = False):

TARGET_ORGANISM_NAME = None
DECODE_STRATEGY = "greedy"
TEMPERATURE = 1.0
TOP_K = 0
TOP_P = 1.0
MAX_DECODE_LEN = params["tgt_len"]

# Classifier-free guidance. 1.0 means normal conditional generation.
CFG_SCALE = 1.0
CLIP_DENOISED = False

# Leveled checkpoints only: bias generation toward potency class
# None = null level (default, organism-only)
# "Weak"=0  "Moderate"=1  "Strong"=2
POTENCY_LEVEL = "Strong"  # "Strong" for more potent bias

LATENT_SCALE = 25.0
NULL_LEVEL = 3  # potency null slot (Weak=0, Moderate=1, Strong=2, null=3)
START_ID = w2i["<start>"]
EOS_ID = w2i["<end>"]
PAD_ID = w2i["_"]

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Device: {DEVICE}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Vocab OK: START={START_ID}, EOS={EOS_ID}, PAD={PAD_ID}")

### Model definitions

Timestep embedding, denoiser (`DModel` with optional level embeddings), and Gaussian diffusion sampler.


In [ ]:
def timestep_embedding(timesteps: torch.Tensor, dim: int, max_period: int = 10000) -> torch.Tensor:
    half = dim // 2
    freqs = torch.exp(
        -math.log(max_period) * torch.arange(start=0, end=half, dtype=torch.float32, device=timesteps.device) / half
    )
    args = timesteps[:, None].float() * freqs[None]
    embedding = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
    if dim % 2:
        embedding = torch.cat([embedding, torch.zeros_like(embedding[:, :1])], dim=-1)
    return embedding


class DModel(nn.Module):

    def __init__(self, num_class: int = 2, num_levels: int = 0, max_length: int = 1):
        super().__init__()
        self.input_channels = 128
        self.model_channels = 127
        self.out_channels = 128
        self.num_class = num_class
        self.num_levels = num_levels
        self.drop_out = 0.1
        self.max_length = max_length

        config = BertConfig()
        config.hidden_dropout_prob = self.drop_out
        config.max_position_embeddings = self.max_length
        config.num_attention_heads = 6
        config.num_hidden_layers = 3
        config._attn_implementation = "eager"

        self.control_embedding = nn.Embedding(self.num_class, config.hidden_size)
        self.level_embedding = (
            nn.Embedding(num_levels, config.hidden_size) if num_levels > 0 else None
        )
        self.time_embed = nn.Sequential(
            nn.Linear(self.model_channels, config.hidden_size),
            nn.SiLU(),
            nn.Linear(config.hidden_size, config.hidden_size),
        )
        self.input_up_proj = nn.Sequential(
            nn.Linear(self.input_channels, config.hidden_size),
            nn.Tanh(),
            nn.Linear(config.hidden_size, config.hidden_size),
        )
        self.input_transformers = BertEncoder(config)
        self.register_buffer("position_ids", torch.arange(config.max_position_embeddings).expand((1, -1)))
        self.position_embeddings = nn.Embedding(config.max_position_embeddings, config.hidden_size)
        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.output_down_proj = nn.Sequential(
            nn.Linear(config.hidden_size, config.hidden_size),
            nn.Tanh(),
            nn.Linear(config.hidden_size, self.out_channels),
        )

    def forward(self, x: torch.Tensor, timesteps: torch.Tensor, control: Optional[torch.Tensor] = None, level: Optional[torch.Tensor] = None) -> torch.Tensor:
        emb = self.time_embed(timestep_embedding(timesteps, self.model_channels))
        if control is not None:
            emb = emb + self.control_embedding(control)
        if level is not None and self.level_embedding is not None:
            emb = emb + self.level_embedding(level)

        emb_x = self.input_up_proj(x)
        seq_length = x.size(1)
        pos_ids = self.position_ids[:, :seq_length]
        emb_inputs = self.position_embeddings(pos_ids) + emb_x + emb.unsqueeze(1).expand(-1, seq_length, -1)
        emb_inputs = self.dropout(self.LayerNorm(emb_inputs))

        attn_mask = torch.ones(x.shape[0], seq_length, device=x.device)
        extended_mask = attn_mask[:, None, None, :]
        extended_mask = (1.0 - extended_mask) * torch.finfo(emb_inputs.dtype).min
        h = self.input_transformers(emb_inputs, attention_mask=extended_mask)[0]
        return self.output_down_proj(h).type(x.dtype)


class CFGDenoiser(nn.Module):
    def __init__(self, model: nn.Module, cfg_scale: float, null_label: Optional[int]):
        super().__init__()
        self.model = model
        self.cfg_scale = cfg_scale
        self.null_label = null_label

    def forward(self, x: torch.Tensor, timesteps: torch.Tensor, control: Optional[torch.Tensor] = None, level: Optional[torch.Tensor] = None) -> torch.Tensor:
        if control is None or self.cfg_scale == 1.0 or self.null_label is None:
            return self.model(x, timesteps, control, level)
        null_control = torch.full_like(control, self.null_label)
        cond_out = self.model(x, timesteps, control, level)
        uncond_out = self.model(x, timesteps, null_control, level)
        return uncond_out + self.cfg_scale * (cond_out - uncond_out)


def _extract_into_tensor(arr: np.ndarray, timesteps: torch.Tensor, broadcast_shape: Tuple[int, ...]) -> torch.Tensor:
    res = torch.from_numpy(arr).to(device=timesteps.device)[timesteps].float()
    while len(res.shape) < len(broadcast_shape):
        res = res[..., None]
    return res.expand(broadcast_shape)


class MyDiffusion:
    def __init__(self, num_timesteps: int, betas: np.ndarray):
        self.betas = betas
        alphas = 1.0 - betas
        self.num_timesteps = num_timesteps
        self.alphas_cumprod = np.cumprod(alphas, axis=0)
        self.alphas_cumprod_prev = np.append(1.0, self.alphas_cumprod[:-1])
        self.posterior_variance = betas * (1.0 - self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)
        self.posterior_log_variance_clipped = np.log(np.append(self.posterior_variance[1], self.posterior_variance[1:]))
        self.posterior_mean_coef1 = betas * np.sqrt(self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)
        self.posterior_mean_coef2 = (
            (1.0 - self.alphas_cumprod_prev) * np.sqrt(alphas) / (1.0 - self.alphas_cumprod)
        )

    def _scale_timesteps(self, t: torch.Tensor) -> torch.Tensor:
        return t.float() * (1000.0 / self.num_timesteps)

    def q_posterior_mean_variance(self, x_start: torch.Tensor, x_t: torch.Tensor, t: torch.Tensor):
        mean = (
            _extract_into_tensor(self.posterior_mean_coef1, t, x_t.shape) * x_start
            + _extract_into_tensor(self.posterior_mean_coef2, t, x_t.shape) * x_t
        )
        log_var = _extract_into_tensor(self.posterior_log_variance_clipped, t, x_t.shape)
        return mean, log_var

    def p_sample(self, model: nn.Module, x: torch.Tensor, t: torch.Tensor, cond=None, level=None, clip_denoised: bool = False):
        model_output = model(x, self._scale_timesteps(t), control=cond, level=level)
        pred_xstart = model_output.clamp(-1, 1) if clip_denoised else model_output
        mean, log_var = self.q_posterior_mean_variance(pred_xstart, x, t)
        noise = torch.randn_like(x)
        nonzero_mask = (t != 0).float().view(-1, *([1] * (len(x.shape) - 1)))
        return mean + nonzero_mask * torch.exp(0.5 * log_var) * noise

    @torch.no_grad()
    def p_sample_loop(self, model: nn.Module, shape: Tuple[int, ...], cond, device: torch.device, level=None, clip_denoised: bool = False):
        img = torch.randn(*shape, device=device)
        for i in tqdm(range(self.num_timesteps - 1, -1, -1), desc="diffusion", leave=False):
            t = torch.full((shape[0],), i, dtype=torch.long, device=device)
            img = self.p_sample(model, img, t, cond=cond, level=level, clip_denoised=clip_denoised)
        return img


def betas_for_alpha_bar(num_diffusion_timesteps: int, alpha_bar, max_beta: float = 0.999) -> np.ndarray:
    betas = []
    for i in range(num_diffusion_timesteps):
        t1 = i / num_diffusion_timesteps
        t2 = (i + 1) / num_diffusion_timesteps
        betas.append(min(1 - alpha_bar(t2) / alpha_bar(t1), max_beta))
    return np.array(betas)


def create_gaussian_diffusion(steps: int = 500) -> MyDiffusion:
    betas = betas_for_alpha_bar(steps, lambda t: 1 - np.sqrt(t + 0.0001))
    return MyDiffusion(num_timesteps=steps, betas=betas)

### Checkpoint loading and decoding

Load VAE + diffusion weights, CFG sampling, and greedy/sample peptide decoding.


In [ ]:
from src.utils import strip_module_prefix, load_checkpoint

def load_vae(path: str, device: torch.device) -> nn.Module:
    model = create_VAE()
    state_dict, _ = load_checkpoint(path, device)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return model


def ids_to_peptides(token_ids: torch.Tensor) -> List[str]:
    peptides = []
    for row in token_ids.detach().cpu().long().tolist():
        chars = []
        for token_id in row:
            if token_id in (EOS_ID, PAD_ID):
                break
            token = org_dict.get(int(token_id) - 1)
            if token is None or token == "<start>":
                continue
            chars.append(token)
        peptides.append("".join(chars))
    return peptides


def choose_next_token(logits: torch.Tensor, strategy: str, temperature: float, top_k: int, top_p: float) -> torch.Tensor:
    if strategy == "greedy":
        return torch.argmax(logits, dim=-1) + 1

    logits = logits / max(temperature, 1e-8)
    if top_k > 0:
        values, _ = torch.topk(logits, min(top_k, logits.shape[-1]), dim=-1)
        logits = logits.masked_fill(logits < values[:, [-1]], -float("inf"))

    if top_p < 1.0:
        sorted_logits, sorted_idx = torch.sort(logits, descending=True, dim=-1)
        sorted_probs = F.softmax(sorted_logits, dim=-1)
        cumulative = torch.cumsum(sorted_probs, dim=-1)
        remove = cumulative > top_p
        remove[:, 1:] = remove[:, :-1].clone()
        remove[:, 0] = False
        sorted_logits = sorted_logits.masked_fill(remove, -float("inf"))
        logits = torch.full_like(logits, -float("inf")).scatter(-1, sorted_idx, sorted_logits)

    probs = F.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1).squeeze(1) + 1


@torch.no_grad()
def decode_latents(vae: nn.Module, latents: torch.Tensor, src_mask: torch.Tensor) -> torch.Tensor:
    batch_size = latents.shape[0]
    decoded = torch.full((batch_size, 1), START_ID, dtype=torch.long, device=latents.device)
    output = torch.full((batch_size, MAX_DECODE_LEN), PAD_ID, dtype=torch.long, device=latents.device)
    finished = torch.zeros(batch_size, dtype=torch.bool, device=latents.device)

    for i in range(MAX_DECODE_LEN):
        tgt_mask = subsequent_mask(decoded.size(1)).to(device=latents.device, dtype=torch.bool)
        logits = vae.generator(vae.decode(latents, src_mask, decoded, tgt_mask))[:, i, :]
        next_token = choose_next_token(logits, DECODE_STRATEGY, TEMPERATURE, TOP_K, TOP_P)
        next_token = torch.where(finished, torch.full_like(next_token, PAD_ID), next_token)
        output[:, i] = next_token

        finished = finished | (next_token == EOS_ID) | (next_token == PAD_ID)
        decoded = torch.cat([decoded, next_token.unsqueeze(1)], dim=1)
        if bool(finished.all()):
            break

    return output


def predict_src_mask_from_latents(vae: nn.Module, latents: torch.Tensor) -> torch.Tensor:
    pred_len = vae.encoder.predict_len2(vae.encoder.predict_len1(latents))
    lengths = F.softmax(pred_len, dim=-1).argmax(dim=-1)
    lengths = lengths.clamp(min=1, max=params["src_len"] + 1)
    positions = torch.arange(params["src_len"] + 1, device=latents.device).unsqueeze(0)
    return (positions < lengths.unsqueeze(1)).unsqueeze(-2)


def prepare_vae_latents(vae: nn.Module, diffusion_sample: torch.Tensor):
    sample = diffusion_sample / LATENT_SCALE
    latents = sample.squeeze(1)
    if latents.ndim != 2 or latents.shape[-1] != params["d_latent"]:
        raise ValueError(f"Expected diffusion_clean sample shape (B, 1, 128), got {tuple(diffusion_sample.shape)}")
    src_mask = predict_src_mask_from_latents(vae, latents)
    return latents, src_mask


def batched(total: int, batch_size: int):
    for start in range(0, total, batch_size):
        yield min(batch_size, total - start)

### Organism map and generation helpers

Resolve organism ids/names, potency level ids, and `generate_for_condition`.


In [ ]:
from src.utils import DEFAULT_ORGANISM_MAP, LEVEL_NAME_TO_ID, resolve_level_id

def load_organism_map() -> List[str]:
    if os.path.isfile(ORGANISM_MAP_PATH):
        raw = np.load(ORGANISM_MAP_PATH, allow_pickle=True)
        org_map = [str(x) for x in raw.tolist()]
    else:
        org_map = list(DEFAULT_ORGANISM_MAP)
        print(f"Note: {ORGANISM_MAP_PATH} not found — using built-in 9-organism map.")
    if len(org_map) == 1:
        print(
            "WARNING: organism map has only 1 entry. Use organism_map.npy "
        )
    return org_map


def resolve_organism_id(org_map: List[str], organism_id: int = None, name: Optional[str] = None) -> int:
    if name:
        key = str(name).strip().lower()
        for i, org in enumerate(org_map):
            o = org.lower()
            if key == o or key in o or o.endswith(key.replace("_", " ")):
                return i
        raise ValueError(f"Organism '{name}' not found in map: {org_map}")
    if organism_id is None:
        raise ValueError("Set ORGANISM_ID or TARGET_ORGANISM_NAME")
    if organism_id < 0 or organism_id >= len(org_map):
        raise ValueError(f"ORGANISM_ID={organism_id} out of range for {len(org_map)} organisms")
    return int(organism_id)


def load_models():
    device = torch.device(DEVICE)

    if SEED is not None:
        random.seed(SEED)
        np.random.seed(SEED)
        torch.manual_seed(SEED)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(SEED)

    print("Loading VAE...")
    vae = load_vae(VAE_CHECKPOINT, device)

    print("Loading diffusion model...")
    diffusion_state, metadata = load_checkpoint(DIFFUSION_CHECKPOINT, device)
    num_steps = NUM_STEPS or metadata.get("num_steps", 500)
    num_class = metadata.get("num_organisms", 1) + 1
    null_label = metadata.get("null_label")

    if "control_embedding.weight" in diffusion_state:
        num_class = diffusion_state["control_embedding.weight"].shape[0]

    num_levels = metadata.get("num_levels", 0)
    if "level_embedding.weight" in diffusion_state:
        num_levels = int(diffusion_state["level_embedding.weight"].shape[0])
    level_null = metadata.get("level_null")
    if level_null is None and num_levels > 0:
        level_null = NULL_LEVEL

    diffusion_model = DModel(num_class=num_class, num_levels=num_levels, max_length=1).to(device)
    diffusion_model.load_state_dict(diffusion_state, strict=False)
    diffusion_model.eval()

    denoiser = CFGDenoiser(diffusion_model, CFG_SCALE, null_label).to(device).eval()
    diffusion = create_gaussian_diffusion(num_steps)

    print(f"Loaded diffusion: steps={num_steps}, num_class={num_class}, null_label={null_label}, levels={num_levels}, level_null={level_null}")
    return vae, denoiser, diffusion, device, num_levels, level_null


@torch.no_grad()
def generate_for_condition(vae, denoiser, diffusion, device, organism_id: int, num_sequences: Optional[int] = None, num_levels: int = 0, level_null: Optional[int] = None, level_id: Optional[int] = None):
    if num_sequences is None:
        num_sequences = NUM_SEQUENCES

    peptides = []
    for current_batch in tqdm(list(batched(num_sequences, BATCH_SIZE)), desc=f"organism {organism_id}"):
        cond = torch.full((current_batch,), organism_id, dtype=torch.long, device=device)
        level = None
        if num_levels > 0 and level_id is not None:
            level = torch.full((current_batch,), level_id, dtype=torch.long, device=device)
        sample = diffusion.p_sample_loop(
            denoiser,
            (current_batch, 1, params["d_latent"]),
            cond=cond,
            device=device,
            level=level,
            clip_denoised=CLIP_DENOISED,
        )
        latents, src_mask = prepare_vae_latents(vae, sample)
        token_ids = decode_latents(vae, latents, src_mask)
        peptides.extend(ids_to_peptides(token_ids))
    return peptides[:num_sequences]


def save_peptides(peptides: List[str], output_path: str):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as handle:
        for peptide in peptides:
            handle.write(f"{peptide}\n")


def print_summary(name: str, peptides: List[str]):
    nonempty = [p for p in peptides if p]
    lengths = [len(p) for p in nonempty]
    print(f"\n{name}")
    print(f"  Generated: {len(peptides)}")
    print(f"  Nonempty : {len(nonempty)}")
    print(f"  Unique   : {len(set(nonempty))}")
    if lengths:
        print(f"  Lengths  : min={min(lengths)} max={max(lengths)} mean={np.mean(lengths):.2f} std={np.std(lengths):.2f}")
        print("  Preview  :")
        for peptide in nonempty[:5]:
            print(f"    {peptide} (len={len(peptide)})")

### Run generation

Load models and generate peptides for the selected organism(s) at the configured potency level.


In [ ]:
vae, denoiser, diffusion, device, num_levels, level_null = load_models()

gen_level_id = resolve_level_id(num_levels, level_null, POTENCY_LEVEL)
if gen_level_id is not None:
    names = {0: "Weak", 1: "Moderate", 2: "Strong", 3: "null"}
    print(f"Potency level at generation: {names.get(gen_level_id, gen_level_id)} (id={gen_level_id})")
else:
    print("Potency level: off (non-leveled checkpoint)")

org_map = load_organism_map()
print(f"Diffusion organism map ({len(org_map)} entries — index = model condition id):")
for i, name in enumerate(org_map):
    print(f"  [{i}] {name}")

all_results = {}
if GENERATE_ALL_ORGANISMS:
    organism_ids = list(range(len(org_map)))
else:
    organism_ids = [resolve_organism_id(org_map, ORGANISM_ID, TARGET_ORGANISM_NAME)]

for organism_id in organism_ids:
    organism_name = str(org_map[organism_id])
    print(f"\n{'='*60}")
    print(f"Model condition id [{organism_id}]  ->  {organism_name}  ({NUM_SEQUENCES} samples)")
    print(f"{'='*60}")
    peptides = generate_for_condition(vae, denoiser, diffusion, device, organism_id, num_levels=num_levels, level_null=level_null, level_id=gen_level_id)
    all_results[organism_name] = peptides

    safe_name = organism_name.replace(" ", "_").replace("/", "-")
    output_path = f"{OUTPUT_DIR}/{safe_name}.txt"
    save_peptides(peptides, output_path)
    print_summary(organism_name, peptides)
    print(f"  Saved: {output_path}")

print("\nDone.")